import pandas as pd
print("Библиотека pandas загружена!")

In [3]:
file_path = r"C:\Users\chamo\Downloads\Задание (2).xlsx"

df = pd.read_excel(file_path, sheet_name="База", header=0)

df.head()

,Площадка,номер заявки,вид_кредита,Дата регистрации заявки,Дата принятия решения,Решение,Дата подписания документов,Бакет
0,Восток,1.0,ИК,2020-12-25 11:33:47,2021-01-14 05:18:20,Отказать,NaT,0
1,Восток,2.0,КК,2021-01-15 06:36:46,2021-01-25 04:12:27,Отказать,NaT,1-30
2,Восток,3.0,ПК,2021-01-14 20:05:10,2021-01-18 04:27:33,Отказать,NaT,0
3,Восток,4.0,ПК,2021-01-14 20:05:10,2021-01-18 04:27:33,Отказать,NaT,0
4,Восток,5.0,ИК,2021-01-14 20:05:10,2021-01-18 04:27:33,Отказать,NaT,0


In [4]:
df = df[df["номер заявки"].notna()].copy()
print(f"После удаления пустых строк: {len(df)} заявок")

if "Nat" in df.columns:
    df.rename(columns={"Nat": "Бакет"}, inplace=True)
    print("Колонка 'Nat' переименована в 'Бакет'")

df["номер заявки"] = df["номер заявки"].astype(int)

df["Бакет"] = df["Бакет"].astype(str)

df["Неделя регистрации"] = df["Дата регистрации заявки"].dt.to_period("W").dt.start_time
df["Месяц регистрации"] = df["Дата регистрации заявки"].dt.to_period("M").dt.start_time

print(f"\nИтоговое количество строк: {len(df)}")
df.head()

После удаления пустых строк: 11031 заявок

Итоговое количество строк: 11031


,Площадка,номер заявки,вид_кредита,Дата регистрации заявки,Дата принятия решения,Решение,Дата подписания документов,Бакет,Неделя регистрации,Месяц регистрации
0,Восток,1,ИК,2020-12-25 11:33:47,2021-01-14 05:18:20,Отказать,NaT,0,2020-12-21,2020-12-01
1,Восток,2,КК,2021-01-15 06:36:46,2021-01-25 04:12:27,Отказать,NaT,1-30,2021-01-11,2021-01-01
2,Восток,3,ПК,2021-01-14 20:05:10,2021-01-18 04:27:33,Отказать,NaT,0,2021-01-11,2021-01-01
3,Восток,4,ПК,2021-01-14 20:05:10,2021-01-18 04:27:33,Отказать,NaT,0,2021-01-11,2021-01-01
4,Восток,5,ИК,2021-01-14 20:05:10,2021-01-18 04:27:33,Отказать,NaT,0,2021-01-11,2021-01-01


In [5]:
df["Время на решение (часы)"] = (df["Дата принятия решения"] - df["Дата регистрации заявки"]).dt.total_seconds() / 3600

avg_time = df["Время на решение (часы)"].mean()
median_time = df["Время на решение (часы)"].median()

print("=" * 60)
print("ПОКАЗАТЕЛЬ 1: TIME TO DECISION")
print("=" * 60)
print(f"Среднее время на решение: {avg_time:.1f} часов")
print(f"Медианное время на решение: {median_time:.1f} часов")
print(f"Минимальное время: {df['Время на решение (часы)'].min():.1f} часов")
print(f"Максимальное время: {df['Время на решение (часы)'].max():.1f} часов")

ПОКАЗАТЕЛЬ 1: TIME TO DECISION
Среднее время на решение: 100.6 часов
Медианное время на решение: 49.9 часов
Минимальное время: 0.1 часов
Максимальное время: 3099.1 часов


In [5]:
print("\n" + "=" * 60)
print("СРЕЗЫ ПО TIME TO DECISION")
print("=" * 60)

print("\nПо площадкам:")
print(df.groupby("Площадка")["Время на решение (часы)"].mean().round(1))

print("\nПо виду кредита:")
print(df.groupby("вид_кредита")["Время на решение (часы)"].mean().round(1))

print("\nПо бакету:")
print(df.groupby("Бакет")["Время на решение (часы)"].mean().round(1))


СРЕЗЫ ПО TIME TO DECISION

По площадкам:
Площадка
Восток    106.9
Запад      98.3
Name: Время на решение (часы), dtype: float64

По виду кредита:
вид_кредита
ИК    112.5
КК    101.4
ПК     99.0
Name: Время на решение (часы), dtype: float64

По бакету:
Бакет
0          90.9
1-30      109.4
181+      117.7
31-90     112.4
91-180    140.2
Name: Время на решение (часы), dtype: float64


In [6]:
# ============================================
# ПОКАЗАТЕЛЬ 2: УРОВЕНЬ ОДОБРЕНИЯ
# ============================================

print("=" * 60)
print("ПОКАЗАТЕЛЬ 2: УРОВЕНЬ ОДОБРЕНИЯ")
print("=" * 60)

total_approval = (df["Решение"] == "Одобрить").mean() * 100
print(f"Общий уровень одобрения: {total_approval:.1f}%")

print("\nПо площадкам (%):")
print(df.groupby("Площадка")["Решение"].apply(lambda x: (x == "Одобрить").mean() * 100).round(1))

print("\nПо виду кредита (%):")
print(df.groupby("вид_кредита")["Решение"].apply(lambda x: (x == "Одобрить").mean() * 100).round(1))

print("\nПо бакету (%):")
print(df.groupby("Бакет")["Решение"].apply(lambda x: (x == "Одобрить").mean() * 100).round(1))

print("\nПо площадке и бакету (%):")
cross = df.groupby(["Площадка", "Бакет"])["Решение"].apply(lambda x: (x == "Одобрить").mean() * 100).round(1)
print(cross)

ПОКАЗАТЕЛЬ 2: УРОВЕНЬ ОДОБРЕНИЯ
Общий уровень одобрения: 21.0%

По площадкам (%):
Площадка
Восток    19.9
Запад     21.5
Name: Решение, dtype: float64

По виду кредита (%):
вид_кредита
ИК    19.4
КК    20.5
ПК    21.3
Name: Решение, dtype: float64

По бакету (%):
Бакет
0         24.5
1-30      18.9
181+       5.6
31-90     16.7
91-180    12.1
Name: Решение, dtype: float64

По площадке и бакету (%):
Площадка  Бакет 
Восток    0         24.8
          1-30      17.3
          181+       1.6
          31-90      9.7
          91-180     9.0
Запад     0         24.4
          1-30      19.4
          181+       7.0
          31-90     18.7
          91-180    13.4
Name: Решение, dtype: float64


In [7]:
# ============================================
# ПОКАЗАТЕЛЬ 3: КОНВЕРСИЯ РЕШЕНИЙ В СДЕЛКИ
# ============================================

print("=" * 60)
print("ПОКАЗАТЕЛЬ 3: КОНВЕРСИЯ РЕШЕНИЙ В СДЕЛКИ")
print("=" * 60)

approved = df[df["Решение"] == "Одобрить"].copy()

approved["Подписано"] = approved["Дата подписания документов"].notna()
conversion_rate = approved["Подписано"].mean() * 100

print(f"Общая конверсия одобренных в сделки: {conversion_rate:.1f}%")
print(f"Всего одобрено: {len(approved)} заявок")
print(f"Из них подписано: {approved['Подписано'].sum()} заявок")

print("\nПо площадкам (%):")
print(approved.groupby("Площадка")["Подписано"].apply(lambda x: x.mean() * 100).round(1))

print("\nПо виду кредита (%):")
print(approved.groupby("вид_кредита")["Подписано"].apply(lambda x: x.mean() * 100).round(1))

print("\nПо бакету (%):")
print(approved.groupby("Бакет")["Подписано"].apply(lambda x: x.mean() * 100).round(1))

ПОКАЗАТЕЛЬ 3: КОНВЕРСИЯ РЕШЕНИЙ В СДЕЛКИ
Общая конверсия одобренных в сделки: 79.2%
Всего одобрено: 2321 заявок
Из них подписано: 1839 заявок

По площадкам (%):
Площадка
Восток    80.4
Запад     78.8
Name: Подписано, dtype: float64

По виду кредита (%):
вид_кредита
ИК    82.8
КК    41.6
ПК    80.7
Name: Подписано, dtype: float64

По бакету (%):
Бакет
0         78.7
1-30      77.4
181+      64.3
31-90     89.9
91-180    69.1
Name: Подписано, dtype: float64


In [8]:
# ============================================
# ПОКАЗАТЕЛЬ 4: ВХОДЯЩИЙ ПОТОК ПО МЕСЯЦАМ
# ============================================

print("=" * 60)
print("ПОКАЗАТЕЛЬ 4: ВХОДЯЩИЙ ПОТОК ПО МЕСЯЦАМ/ГОДАМ")
print("=" * 60)

monthly_flow = df.groupby("Месяц регистрации").size()

print("\nКоличество заявок по месяцам:")
print(monthly_flow)

print(f"\nВсего заявок за весь период: {monthly_flow.sum()}")

df["Год_месяц"] = df["Дата регистрации заявки"].dt.to_period("M")
year_month_flow = df.groupby("Год_месяц").size()
print("\nПо годам и месяцам:")
print(year_month_flow)

ПОКАЗАТЕЛЬ 4: ВХОДЯЩИЙ ПОТОК ПО МЕСЯЦАМ/ГОДАМ

Количество заявок по месяцам:
Месяц регистрации
2020-09-01      14
2020-10-01       5
2020-11-01      10
2020-12-01     624
2021-01-01    8312
2021-02-01    2066
dtype: int64

Всего заявок за весь период: 11031

По годам и месяцам:
Год_месяц
2020-09      14
2020-10       5
2020-11      10
2020-12     624
2021-01    8312
2021-02    2066
Freq: M, dtype: int64


In [9]:
# ============================================
# ПОКАЗАТЕЛЬ 5: ПРОГНОЗ НА МАРТ-МАЙ 2021
# ============================================

print("=" * 60)
print("ПОКАЗАТЕЛЬ 5: ПРОГНОЗ ВХОДЯЩЕГО ПОТОКА")
print("=" * 60)


feb_data = df[df["Месяц регистрации"] == "2021-02-01"]
days_in_feb = 28
days_in_sample = 10

avg_daily_feb = len(feb_data) / days_in_sample
estimated_full_feb = avg_daily_feb * days_in_feb

print(f"Февраль (данные): {len(feb_data)} заявок за {days_in_sample} дней")
print(f"Среднедневной поток в феврале: {avg_daily_feb:.1f} заявок в день")
print(f"Оценка полного февраля: {estimated_full_feb:.0f} заявок")


dec_flow = monthly_flow["2020-12-01"]
jan_flow = monthly_flow["2021-01-01"]
feb_flow = estimated_full_feb

months = ["Декабрь", "Январь", "Февраль (оценка)"]
values = [dec_flow, jan_flow, feb_flow]

print("\nИсходные данные для прогноза:")
for m, v in zip(months, values):
    print(f"  {m}: {v:.0f}")


growth_dec_jan = (jan_flow - dec_flow) / dec_flow
growth_jan_feb = (feb_flow - jan_flow) / jan_flow
avg_growth = (growth_dec_jan + growth_jan_feb) / 2

print(f"\nТемп роста: декабрь→январь: {growth_dec_jan:.1%}, январь→февраль: {growth_jan_feb:.1%}")
print(f"Средний темп роста: {avg_growth:.1%}")

forecast_months = ["Март 2021", "Апрель 2021", "Май 2021"]
forecast_values = []
current = feb_flow

for _ in range(3):
    current = current * (1 + avg_growth)
    forecast_values.append(current)

print("\n" + "=" * 60)
print("ПРОГНОЗ ВХОДЯЩЕГО ПОТОКА:")
print("=" * 60)
for month, value in zip(forecast_months, forecast_values):
    print(f"{month}: {value:.0f} заявок")

ПОКАЗАТЕЛЬ 5: ПРОГНОЗ ВХОДЯЩЕГО ПОТОКА
Февраль (данные): 2066 заявок за 10 дней
Среднедневной поток в феврале: 206.6 заявок в день
Оценка полного февраля: 5785 заявок

Исходные данные для прогноза:
  Декабрь: 624
  Январь: 8312
  Февраль (оценка): 5785

Темп роста: декабрь→январь: 1232.1%, январь→февраль: -30.4%
Средний темп роста: 600.8%

ПРОГНОЗ ВХОДЯЩЕГО ПОТОКА:
Март 2021: 40541 заявок
Апрель 2021: 284123 заявок
Май 2021: 1991198 заявок


In [10]:
# ============================================
# УТОЧНЕННЫЙ ПРОГНОЗ (БЕЗ ЯНВАРЯ)
# ============================================

print("=" * 60)
print("УТОЧНЕННЫЙ ПРОГНОЗ (С УЧЕТОМ ТОЛЬКО СТАБИЛЬНЫХ МЕСЯЦЕВ)")
print("=" * 60)

dec_flow = 624
feb_flow = 5785  

growth_dec_feb = (feb_flow - dec_flow) / dec_flow

monthly_growth = (1 + growth_dec_feb) ** (1/2) - 1

print(f"Рост с декабря по февраль: {growth_dec_feb:.1%}")
print(f"Среднемесячный рост: {monthly_growth:.1%}")

current = feb_flow
forecast_months = ["Март 2021", "Апрель 2021", "Май 2021"]
forecast_values = []

for month in forecast_months:
    current = current * (1 + monthly_growth)
    forecast_values.append(current)
    print(f"{month}: {current:.0f} заявок")

print("\n" + "=" * 60)
print("АЛЬТЕРНАТИВНЫЙ ПРОГНОЗ (по среднему значению)")
print("=" * 60)

avg_flow = (dec_flow + feb_flow) / 2
print(f"Среднемесячный поток (декабрь + февраль): {avg_flow:.0f} заявок")

for month in forecast_months:
    print(f"{month}: {avg_flow:.0f} заявок (без учета роста)")

УТОЧНЕННЫЙ ПРОГНОЗ (С УЧЕТОМ ТОЛЬКО СТАБИЛЬНЫХ МЕСЯЦЕВ)
Рост с декабря по февраль: 827.1%
Среднемесячный рост: 204.5%
Март 2021: 17614 заявок
Апрель 2021: 53632 заявок
Май 2021: 163298 заявок

АЛЬТЕРНАТИВНЫЙ ПРОГНОЗ (по среднему значению)
Среднемесячный поток (декабрь + февраль): 3204 заявок
Март 2021: 3204 заявок (без учета роста)
Апрель 2021: 3204 заявок (без учета роста)
Май 2021: 3204 заявок (без учета роста)


In [11]:
# ============================================
# ИТОГОВЫЙ ПРОГНОЗ С ДИАПАЗОНАМИ
# ============================================

print("=" * 60)
print("ИТОГОВЫЙ ПРОГНОЗ С ДИАПАЗОНАМИ")
print("=" * 60)

base_forecast = 3204

optimistic = base_forecast * 1.20
pessimistic = base_forecast * 0.80

print(f"Базовый прогноз: {base_forecast} заявок в месяц")
print(f"Оптимистичный (+20%): {optimistic:.0f} заявок в месяц")
print(f"Пессимистичный (-20%): {pessimistic:.0f} заявок в месяц")

print("\n" + "=" * 60)
print("ПРОГНОЗ НА МАРТ-МАЙ 2021:")
print("=" * 60)

months = ["Март 2021", "Апрель 2021", "Май 2021"]

print(f"{'Месяц':<15} {'Базовый':<12} {'Оптимистичный':<15} {'Пессимистичный'}")
print("-" * 60)
for month in months:
    print(f"{month:<15} {base_forecast:<12} {optimistic:<15.0f} {pessimistic:<.0f}")

ИТОГОВЫЙ ПРОГНОЗ С ДИАПАЗОНАМИ
Базовый прогноз: 3204 заявок в месяц
Оптимистичный (+20%): 3845 заявок в месяц
Пессимистичный (-20%): 2563 заявок в месяц

ПРОГНОЗ НА МАРТ-МАЙ 2021:
Месяц           Базовый      Оптимистичный   Пессимистичный
------------------------------------------------------------
Март 2021       3204         3845            2563
Апрель 2021     3204         3845            2563
Май 2021        3204         3845            2563


# Анализ процесса реструктуризации кредитов

## 1. Time to Decision (время на решение)

- **Среднее время:** 100.6 часов (~4 дня)
- **Медианное время:** 49.9 часов (~2 дня)
- **Минимум:** 0.1 часа (6 минут)
- **Максимум:** 3099 часов (~129 дней)

**Вывод:** половина заявок рассматривается быстрее чем за 2 дня, однако есть небольшое количество очень долгих заявок, которые сильно увеличивают среднее значение.

### Срезы:

| Срез | Значение |
|------|----------|
| **Площадка** | Восток — 106.9 ч, Запад — 98.3 ч |
| **Вид кредита** | ИК — 112.5 ч, КК — 101.4 ч, ПК — 99.0 ч |
| **Бакет** | 0 — 90.9 ч, 1-30 — 109.4 ч, 31-90 — 112.4 ч, 91-180 — 140.2 ч, 181+ — 117.7 ч |

**Ключевой вывод:** чем выше просрочка, тем дольше рассматривают заявку. Бакет 91-180 — самый долгий (140.2 ч).

---

## 2. Уровень одобрения

- **Общий уровень одобрения:** 21.0%

### Срезы:

| Срез | Значение |
|------|----------|
| **Площадка** | Восток — 19.9%, Запад — 21.5% |
| **Вид кредита** | ИК — 19.4%, КК — 20.5%, ПК — 21.3% |
| **Бакет** | 0 — 24.5%, 1-30 — 18.9%, 31-90 — 16.7%, 91-180 — 12.1%, 181+ — 5.6% |

**Ключевой вывод:** наличие просрочки (бакет) — главный фактор отказа. Для бакета 181+ одобрение в 4 раза ниже, чем для бакета 0.

---

## 3. Конверсия решений в сделки

- **Общая конверсия:** 79.2% (из 2321 одобренных подписано 1839)

### Срезы:

| Срез | Значение |
|------|----------|
| **Площадка** | Восток — 80.4%, Запад — 78.8% |
| **Вид кредита** | ИК — 82.8%, КК — 41.6%, ПК — 80.7% |
| **Бакет** | 0 — 78.7%, 1-30 — 77.4%, 31-90 — 89.9%, 91-180 — 69.1%, 181+ — 64.3% |

**Ключевой вывод:** кредитные карты (КК) показывают аномально низкую конверсию — только 41.6% одобренных заявок доходят до подписания.

---

## 4. Входящий поток по месяцам

| Месяц | Заявки |
|-------|--------|
| Сентябрь 2020 | 14 |
| Октябрь 2020 | 5 |
| Ноябрь 2020 | 10 |
| Декабрь 2020 | 624 |
| Январь 2021 | 8 312 |
| Февраль 2021 | 2 066 (данные за 10 дней) |

**Ключевой вывод:** январь 2021 — аномальный пик (75% всех заявок). Февраль в данных неполный.

---

## 5. Прогноз на март-май 2021

**Метод:** среднемесячный поток за стабильные месяцы (декабрь + февраль). Январь исключен как аномалия.

| Сценарий | Март | Апрель | Май |
|----------|------|--------|-----|
| Базовый | 3 204 | 3 204 | 3 204 |
| Оптимистичный (+20%) | 3 845 | 3 845 | 3 845 |
| Пессимистичный (-20%) | 2 563 | 2 563 | 2 563 |

**Допущения:**
1. В марте-мае не будет повторения январского аномального пика.
2. Сезонный фактор не окажет значительного влияния.
3. Внешние факторы (акции, изменение условий) не учитывались.

**Рекомендации:**
- Использовать базовый прогноз для планирования ресурсов.
- Учитывать пессимистичный сценарий для оценки рисков.
- Пересмотреть прогноз в апреле на основе фактических данных за март.

In [12]:
output_path = r"C:\Users\chamo\Downloads\data_for_dashboard.xlsx"
df.to_excel(output_path, index=False)
print(f"Данные сохранены: {output_path}")

Данные сохранены: C:\Users\chamo\Downloads\data_for_dashboard.xlsx


In [13]:
import pandas as pd

file_path = r"C:\Users\chamo\Downloads\Задание (2).xlsx"
df = pd.read_excel(file_path, sheet_name="База", header=0)

df = df[df["номер заявки"].notna()].copy()
if "Nat" in df.columns:
    df.rename(columns={"Nat": "Бакет"}, inplace=True)
df["номер заявки"] = df["номер заявки"].astype(int)
df["Бакет"] = df["Бакет"].astype(str)

df["Неделя регистрации"] = df["Дата регистрации заявки"].dt.to_period("W").dt.start_time
df["Месяц регистрации"] = df["Дата регистрации заявки"].dt.to_period("M").dt.start_time

df["Время на решение (часы)"] = (df["Дата принятия решения"] - df["Дата регистрации заявки"]).dt.total_seconds() / 3600

dashboard_df = df[[
    "Площадка", 
    "вид_кредита", 
    "Бакет", 
    "Решение", 
    "Дата подписания документов",
    "Время на решение (часы)",
    "Неделя регистрации",
    "Месяц регистрации"
]].copy()

dashboard_df["Одобрено"] = (dashboard_df["Решение"] == "Одобрить").astype(int)
dashboard_df["Подписано"] = dashboard_df["Дата подписания документов"].notna().astype(int)

output_path = r"C:\Users\chamo\Downloads\dashboard_data.xlsx"
dashboard_df.to_excel(output_path, index=False)

print(f"✅ Готово! Файл сохранен: {output_path}")
print(f"   Всего строк: {len(dashboard_df)}")
print(f"   Колонки: {list(dashboard_df.columns)}")

✅ Готово! Файл сохранен: C:\Users\chamo\Downloads\dashboard_data.xlsx
   Всего строк: 11031
   Колонки: ['Площадка', 'вид_кредита', 'Бакет', 'Решение', 'Дата подписания документов', 'Время на решение (часы)', 'Неделя регистрации', 'Месяц регистрации', 'Одобрено', 'Подписано']


##  Ключевые бизнес-инсайты и рекомендации

### 1. Просрочка — главный драйвер затрат и риска
Заявки с просрочкой 91–180 дней требуют на 54% больше времени на рассмотрение и одобряются в 2 раза реже.  
**Рекомендация:** внедрить отдельный поток для сложных заявок.

### 2. Кредитные карты — слабое звено в воронке
Конверсия одобренных заявок по КК в 2 раза ниже, чем по другим продуктам.  
**Рекомендация:** проверить условия и клиентский опыт на этапе подписания.

### 3. Сезонный всплеск требует планирования
На январь пришлось 75% всех заявок — это требует пересмотра ресурсного планирования.  
**Рекомендация:** усилить команду в пиковые месяцы и автоматизировать часть процессов.